# Lab 3 — Event Hub Producer
**Goal:** generate synthetic streaming events and publish them to the cohort's shared Azure Event Hub, so the companion `Lab3_EventHub_Consumer` notebook has a live source to read via Spark Structured Streaming (Kafka-compatible protocol).

This notebook only produces/sends events — ingestion, parsing, and bronze-layer storage happen in the consumer notebook.

In [0]:
%pip install azure-eventhub

In [0]:
%restart_python

## 1. Event Hub Connection
The connection string carries a `SharedAccessKey` with send rights on the shared, cohort-wide Event Hub — it must never be hardcoded in the notebook source (that would leak it into notebook history, exports, and version control).

Instead it's read from a Databricks **widget**, the same pattern used in the consumer notebook.

In [0]:
dbutils.widgets.text("eventhub_connection_string", "")
connection_string = dbutils.widgets.get("eventhub_connection_string")

eventhub_name = "ayyuborujzade_evh"

## 2. Generate & Send Synthetic Events
Each event is a small JSON payload (`event_id`, `event_type`, `user_id`, UTC `timestamp`) matching the schema the consumer notebook expects. Events are sent one batch at a time via `EventHubProducerClient`, with a short sleep between sends to simulate a steady real-time stream rather than a single burst — this gives the consumer's `availableNow` trigger and micro-batching something realistic to pick up across multiple runs.

In [0]:
from azure.eventhub import EventHubProducerClient, EventData
import json
import random
import time
from datetime import datetime, timezone

NUM_EVENTS = 100
SEND_INTERVAL_SECONDS = 1

producer = EventHubProducerClient.from_connection_string(conn_str=connection_string)

try:
    for i in range(NUM_EVENTS):
        event = {
            "event_id": i,
            "event_type": random.choice(["click", "purchase", "login", "logout"]),
            "user_id": random.randint(1, 1000),
            "timestamp": datetime.now(timezone.utc).isoformat(),
        }

        event_batch = producer.create_batch()
        event_batch.add(EventData(json.dumps(event)))
        producer.send_batch(event_batch)

        print(event)
        time.sleep(SEND_INTERVAL_SECONDS)
finally:
    producer.close()